# Retail forecasting: preprocessing + modelling in one notebook

test_shop + set_a + set_c → gap-free weekly panel → causal features → rolling backtest → blended submission (w1..w13).

**Changes vs the previous two notebooks**
1. **Trained on the metric.** Rows are weighted by `source_weight × scale / mase_den`, so L1 loss on the scaled target equals per-article MASE. `mase_den` is the expanding mean absolute week-to-week change, computed causally, the same way the scorer does.
2. **Features are actually scale-free.** All `lag_*` / `roll_*` columns are divided by `scale`; `log_scale` and `den_scaled` are added so the model knows the article's size and volatility.
3. **Stock features.** `in_stock` is kept (Set A's boolean, `on_hand > 0` for the test shop), plus `stockout_rate_4/13`, `weeks_since_stockout`, `censored_rate_13`, `on_hand_scaled`.
4. **Four backtest windows, including the same season one year earlier**, which counts double in all summaries and in model selection.
5. **Blend + calibration.** Weights over the best models and one multiplier are tuned on the windows; the blend's backtest score is measured *leave-one-window-out* so it competes fairly with single models.
6. **LightGBM picks its own number of trees** on an inner window (early stopping on the MASE-weighted L1).
7. **Diagnostics:** error concentration by article, error by demand class, feature importance.
8. Test-shop panel starts at each article's first observed week (no fake pre-launch zeros); `Y` is built from the gap-filled panel.

In [ ]:
%pip install -q numpy pandas pyarrow scikit-learn lightgbm catboost xgboost matplotlib

## 0. Imports & configuration ← edit here

In [ ]:
import gc
import itertools
import re
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import SGDRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, OrdinalEncoder, StandardScaler

try:
    from catboost import CatBoostRegressor
    HAS_CATBOOST = True
except ImportError:
    HAS_CATBOOST = False
try:
    import xgboost as xgb
    HAS_XGB = True
except ImportError:
    HAS_XGB = False

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 200)

IN_COLAB = Path('/content').exists()
OUT_DIR = Path('/content/processed') if IN_COLAB else Path('processed')
SUBMISSION_DIR = Path('/content/submissions') if IN_COLAB else Path('submissions')

H = 13
SID = 'series_id'
SEED = 0

# --- data ---
SOURCE_WEIGHTS = {'test_shop': 1.0, 'set_a': 0.35, 'set_c': 0.35}
EXPLICIT_PATHS = {'test_shop': None, 'set_a': None, 'set_c': None}   # optional overrides
SAVE_FEATURES = False                                                 # write features_wide.parquet

# --- backtest ---
FAST_MODE = True                     # False: denser origins + hist_gbdt + linear_l1
N_WINDOWS = 4                        # 4 = includes the same-season window one year back
SAME_SEASON_WEIGHT = 2.0             # weight of that window in every summary / selection
ORIGIN_STRIDE = {'test_shop': 2, 'set_a': 6, 'set_c': 6} if FAST_MODE else {'test_shop': 1, 'set_a': 4, 'set_c': 4}
SELECT_METRIC = 'w8-w13'             # 'all', 'w1-w7' or 'w8-w13': what the final ranking uses

# --- MASE weighting ---
DEN_FLOOR = 0.05                     # floor on mase_den so near-constant articles don't dominate
WEIGHT_CAP_Q = 0.995                 # cap extreme weights at this quantile

# --- models ---
LGB_ROUNDS = 'auto'                  # 'auto' = early stopping on an inner window, or an int
USE_XGB = HAS_XGB                    # XGBoost L1; runs on the GPU if CUDA works, else CPU

BLUE, GREY, INK = '#2a78d6', '#b5b4ae', '#52514e'
plt.rcParams.update({
    'figure.figsize': (11, 4), 'figure.dpi': 100,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.edgecolor': GREY, 'axes.labelcolor': INK, 'xtick.color': INK, 'ytick.color': INK,
    'axes.grid': True, 'grid.color': '#e6e5e0', 'grid.linewidth': 0.8, 'axes.axisbelow': True,
    'axes.titlesize': 12, 'axes.titleweight': 'bold', 'axes.titlelocation': 'left',
})

XGB_DEVICE = 'cpu'
if USE_XGB:
    try:
        xgb.XGBRegressor(device='cuda', n_estimators=2, tree_method='hist').fit(np.random.rand(64, 3), np.random.rand(64))
        XGB_DEVICE = 'cuda'
    except Exception:
        XGB_DEVICE = 'cpu'
print(f'Mode: {"FAST" if FAST_MODE else "FULL"} | windows={N_WINDOWS} | CatBoost={HAS_CATBOOST} | XGBoost={USE_XGB} ({XGB_DEVICE})')

## 1. Locate & load test_shop, set_a, set_c

Searches `/content/` (Colab) or `.`, `./data/`, `./data/hackathon-retail-sets/` for `history.parquet`, `set-a`, `set-c` (parquet or csv), or uses `EXPLICIT_PATHS`.

In [ ]:
SEARCH_ROOTS = [Path('/content'), Path('.')] if IN_COLAB else [Path('.'), Path('data'), Path('data/hackathon-retail-sets')]


def parse_week_start(s):
    if pd.api.types.is_numeric_dtype(s):
        return pd.to_datetime(s, unit='D', origin='1970-01-01')
    return pd.to_datetime(s)


def _read_table(path):
    p = Path(path)
    if p.suffix.lower() == '.parquet':
        return pd.read_parquet(p, engine='pyarrow')
    if p.suffix.lower() == '.csv':
        return pd.read_csv(p)
    raise ValueError(f'Unsupported file extension: {p}')


def discover_dataset_files(roots=SEARCH_ROOTS):
    all_files = []
    for r in roots:
        if r.exists():
            for ext in ('*.parquet', '*.csv'):
                all_files.extend([p for p in r.rglob(ext)
                                  if p.stat().st_size > 0 and 'processed' not in p.parts and 'submissions' not in p.parts])
    found = {'test_shop': [], 'set_a': [], 'set_c': []}
    for p in sorted(set(p.resolve() for p in all_files)):
        low = str(p).lower().replace('\\', '/')
        name = p.name.lower()
        if name.startswith(('features_wide', 'submission')):
            continue
        if 'history' in name or 'test-shop' in low or 'test_shop' in low:
            found['test_shop'].append(p)
        elif any(tok in low for tok in ('set-a', 'set_a', '/seta', 'seta.')):
            found['set_a'].append(p)
        elif any(tok in low for tok in ('set-c', 'set_c', '/setc', 'setc.')):
            found['set_c'].append(p)
    resolved = {}
    for key in found:
        if EXPLICIT_PATHS[key] is not None:
            p = Path(EXPLICIT_PATHS[key])
            resolved[key] = [p] if p.is_file() else sorted(p.rglob('*.parquet')) + sorted(p.rglob('*.csv'))
        else:
            resolved[key] = found[key]
    return resolved


FILES = discover_dataset_files()
for k, v in FILES.items():
    print(f'{k:<10s}: {[str(p) for p in v] if v else "NOT FOUND"}')
missing = [k for k, v in FILES.items() if not v]
if missing:
    raise FileNotFoundError(f'Missing files for {missing}. Put history.parquet, set-a and set-c in ./data or set EXPLICIT_PATHS.')

## 2. Standardize into one weekly panel

Shared schema per row: `source`, `series_id`, `store_id`, `article_id`, `week_start` (Monday), `units`, plus `on_hand` / `in_stock` / `unit_price` / `discount_pct` where a source has them.

- `in_stock`: Set A's boolean as 0/1; for sources with only `on_hand`, derived as `on_hand > 0`.
- Each series runs from its **first observed week** to its last week (test_shop: to the shop's last week, so every article has a row at the forecast origin). Gaps inside the span are filled with 0 units.
- `censored`: 0 units in a week that started with no stock (previous week out of stock).

In [ ]:
DATE_CANDIDATES = ['week_start', 'week', 'date', 'ds', 'timestamp', 'period_start']
UNIT_CANDIDATES = ['units', 'qty', 'quantity', 'sales', 'demand', 'y', 'units_sold']
ITEM_CANDIDATES = ['article_id', 'item_id', 'sku', 'product_id', 'id', 'series_id']
STORE_CANDIDATES = ['store_id', 'shop_id', 'location_id', 'branch_id', 'store']
EXTRA_COLS = ['on_hand', 'in_stock', 'unit_price', 'discount_pct']


def _pick_col(cols, candidates, required=True, label=''):
    low_map = {c.lower(): c for c in cols}
    for cand in candidates:
        if cand in low_map:
            return low_map[cand]
    if required:
        raise KeyError(f'Could not find {label} column in {list(cols)} (tried {candidates})')
    return None


def _to_float(s):
    if s.dtype == bool:
        return s.astype(np.float32)
    if s.dtype == object or pd.api.types.is_string_dtype(s):
        txt = s.astype(str).str.strip().str.lower().map({'true': 1.0, 'false': 0.0, 'yes': 1.0, 'no': 0.0})
        num = pd.to_numeric(s.astype(str).str.strip(), errors='coerce')
        return num.fillna(txt).astype(np.float32)
    return pd.to_numeric(s, errors='coerce').astype(np.float32)


def load_and_standardize_source(source_name, paths, drop_all_zero_series=False):
    t0 = time.perf_counter()
    ts_tables = []
    for p in paths:
        df = _read_table(p)
        cols_low = {c.lower() for c in df.columns}
        if any(c in cols_low for c in DATE_CANDIDATES) and any(c in cols_low for c in UNIT_CANDIDATES):
            ts_tables.append(df)
    if not ts_tables:
        raise ValueError(f'No time-series file with date & units columns found for {source_name} in {paths}')

    raw = pd.concat(ts_tables, ignore_index=True)
    date_col = _pick_col(raw.columns, DATE_CANDIDATES, label='date')
    unit_col = _pick_col(raw.columns, UNIT_CANDIDATES, label='units')
    item_col = _pick_col(raw.columns, ITEM_CANDIDATES, label='article_id')
    store_col = _pick_col(raw.columns, STORE_CANDIDATES, required=False)

    df = pd.DataFrame({
        'article_id': raw[item_col].astype(str),
        'week_start': parse_week_start(raw[date_col]).dt.to_period('W-SUN').dt.start_time,
        'units': pd.to_numeric(raw[unit_col], errors='coerce').fillna(0.0).clip(lower=0.0).astype(np.float32),
    })
    if store_col is not None:
        df['store_id'] = raw[store_col].astype(str)
        df[SID] = source_name + '|' + df['store_id'] + '|' + df['article_id']
    else:
        df['store_id'] = 'default'
        df[SID] = source_name + '|' + df['article_id']

    low_map = {c.lower(): c for c in raw.columns}
    extras = [c for c in EXTRA_COLS if c in low_map]
    for c in extras:
        df[c] = _to_float(raw[low_map[c]])

    agg = {'article_id': 'first', 'store_id': 'first', 'units': 'sum', **{c: 'mean' for c in extras}}
    df = df.groupby([SID, 'week_start'], as_index=False).agg(agg)

    if 'in_stock' not in df.columns and 'on_hand' in df.columns:
        df['in_stock'] = np.where(df['on_hand'].notna(), (df['on_hand'] > 0).astype(np.float32), np.nan).astype(np.float32)

    if drop_all_zero_series:
        df = df[df.groupby(SID)['units'].transform('sum') > 0].reset_index(drop=True)

    # Weekly grid: first observed week -> last week (test_shop: shop's last week for every article)
    first = df.groupby(SID)['week_start'].min()
    last = df.groupby(SID)['week_start'].max()
    if source_name == 'test_shop':
        last[:] = df['week_start'].max()
    grid = pd.DataFrame({SID: first.index,
                         'week_start': [pd.date_range(a, b, freq='W-MON') for a, b in zip(first, last)]}).explode('week_start')
    grid['week_start'] = pd.to_datetime(grid['week_start'])

    id_map = df.groupby(SID, as_index=False)[['article_id', 'store_id']].first()
    grid = grid.merge(id_map, on=SID, how='left')
    dyn = [SID, 'week_start', 'units'] + [c for c in EXTRA_COLS if c in df.columns]
    panel = grid.merge(df[dyn], on=[SID, 'week_start'], how='left')
    panel['observed'] = panel['units'].notna()
    panel['units'] = panel['units'].fillna(0.0).clip(lower=0.0).astype(np.float32)
    panel['source'] = source_name

    if 'in_stock' in panel.columns:
        prev_in = panel.groupby(SID, sort=False)['in_stock'].shift(1)
        panel['censored'] = (panel['units'] == 0) & (prev_in == 0)
    else:
        panel['censored'] = False

    panel = panel.sort_values([SID, 'week_start']).reset_index(drop=True)
    print(f'Loaded {source_name:<10s} in {time.perf_counter() - t0:.1f}s: {len(panel):,} rows | '
          f'{panel[SID].nunique():,} series | {panel["week_start"].min().date()} -> {panel["week_start"].max().date()} | '
          f'extras: {[c for c in EXTRA_COLS if c in panel.columns]}')
    return panel


panel = pd.concat([
    load_and_standardize_source('test_shop', FILES['test_shop'], drop_all_zero_series=False),
    load_and_standardize_source('set_a', FILES['set_a'], drop_all_zero_series=True),
    load_and_standardize_source('set_c', FILES['set_c'], drop_all_zero_series=True),
], ignore_index=True)
gc.collect()
panel['source'] = pd.Categorical(panel['source'], categories=['test_shop', 'set_a', 'set_c'])
panel = panel.sort_values(['source', SID, 'week_start']).reset_index(drop=True)

In [ ]:
def check_panel(panel):
    assert not panel.duplicated([SID, 'week_start']).any(), 'Duplicate (series_id, week_start) rows'
    diffs = panel.groupby(SID, sort=False)['week_start'].diff().dropna()
    assert (diffs == pd.Timedelta(days=7)).all(), 'Non-7-day spacing detected'
    assert (panel['units'] >= 0).all(), 'Negative units'
    tbl = panel.groupby('source', observed=True).agg(
        rows=('units', 'size'), series=(SID, 'nunique'),
        min_week=('week_start', 'min'), max_week=('week_start', 'max'),
        mean_units=('units', 'mean'), zero_rate=('units', lambda s: (s == 0).mean()),
        gap_filled_share=('observed', lambda s: 1 - s.mean()),
        in_stock_known=('in_stock', lambda s: s.notna().mean()) if 'in_stock' in panel else ('units', lambda s: 0.0),
        censored_share=('censored', 'mean'),
    )
    display(tbl)


check_panel(panel)

## 3. Causal features

Everything at row *t* uses data up to and including week *t* only.

- Lags (1, 2, 3, 4, 8, 13, 26, 52) and rolling mean / median / std / max (4, 8, 13, 26, 52 weeks), **divided by `scale`**
- Intermittency: `weeks_since_sale`, `nz_rate_*`, expanding `adi`, `cv2`, Syntetos–Boylan `demand_class`
- Stock: `in_stock`, `stockout_rate_4/13`, `weeks_since_stockout`, `censored_rate_13`, `on_hand_scaled`
- `mase_den`: expanding mean |Δunits| (the MASE denominator), plus `den_scaled = mase_den / scale`
- `scale = max(roll_mean_13, 1)`, `log_scale`, calendar, `seasonal_h1..13` (same week last year / scale), targets `target_h1..13` (units at t+h / scale)

In [ ]:
def classify_demand(adi, cv2, total_sales):
    conds = [total_sales <= 0, (adi < 1.32) & (cv2 < 0.49), (adi < 1.32) & (cv2 >= 0.49),
             (adi >= 1.32) & (cv2 < 0.49), (adi >= 1.32) & (cv2 >= 0.49)]
    return np.select(conds, ['no_sales', 'smooth', 'erratic', 'intermittent', 'lumpy'], default='no_sales')


def _grolling(s, keys, w, how):
    r = s.groupby(keys, sort=False).rolling(w, min_periods=1)
    return getattr(r, how)().reset_index(level=0, drop=True)


def build_features(panel, horizon=H, source_weights=SOURCE_WEIGHTS):
    t0 = time.perf_counter()
    df = panel.sort_values([SID, 'week_start']).reset_index(drop=True).copy()
    keys = df[SID]
    g = df.groupby(SID, sort=False)

    # lifecycle & recency
    df['age_weeks'] = g.cumcount().astype(np.int16)
    is_sale = df['units'] > 0
    last_sale = pd.Series(np.where(is_sale, df['age_weeks'], np.nan)).groupby(keys, sort=False).ffill().to_numpy()
    df['weeks_since_sale'] = (df['age_weeks'].to_numpy(float) - last_sale).astype(np.float32)

    # lags & rolling stats (raw units for now, scaled below)
    for l in [1, 2, 3, 4, 8, 13, 26, 52]:
        df[f'lag_{l}'] = g['units'].shift(l).astype(np.float32)
    for w in [4, 8, 13, 26, 52]:
        r = g['units'].rolling(w, min_periods=1)
        df[f'roll_mean_{w}'] = r.mean().reset_index(level=0, drop=True).astype(np.float32)
        df[f'roll_median_{w}'] = r.median().reset_index(level=0, drop=True).astype(np.float32)
        df[f'roll_std_{w}'] = r.std().fillna(0.0).reset_index(level=0, drop=True).astype(np.float32)
        df[f'roll_max_{w}'] = r.max().reset_index(level=0, drop=True).astype(np.float32)
    sale_f = is_sale.astype(np.float32)
    for w in [4, 13, 26, 52]:
        df[f'nz_rate_{w}'] = _grolling(sale_f, keys, w, 'mean').astype(np.float32)
    df['ratio_mean_4_13'] = (df['roll_mean_4'] / (df['roll_mean_13'] + 1e-3)).astype(np.float32)
    df['ratio_mean_13_52'] = (df['roll_mean_13'] / (df['roll_mean_52'] + 1e-3)).astype(np.float32)

    # Syntetos-Boylan demand class (expanding, causal)
    exp_n = df['age_weeks'].to_numpy(float) + 1.0
    exp_nz = is_sale.groupby(keys, sort=False).cumsum().to_numpy(float)
    exp_sum = g['units'].cumsum().to_numpy(float)
    pos = df['units'].where(is_sale)
    pos_mean = pos.groupby(keys, sort=False).expanding().mean().reset_index(level=0, drop=True).to_numpy(float)
    pos_std = pos.groupby(keys, sort=False).expanding().std().fillna(0.0).reset_index(level=0, drop=True).to_numpy(float)
    with np.errstate(divide='ignore', invalid='ignore'):
        adi = np.where(exp_nz > 0, exp_n / exp_nz, np.inf)
        cv2 = np.where(pos_mean > 0, (pos_std / pos_mean) ** 2, 0.0)
    df['adi'] = np.where(np.isfinite(adi), adi, 999.0).astype(np.float32)
    df['cv2'] = np.nan_to_num(cv2).astype(np.float32)
    df['demand_class'] = pd.Categorical(classify_demand(adi, cv2, exp_sum),
                                        categories=['smooth', 'erratic', 'intermittent', 'lumpy', 'no_sales'])

    # calendar
    woy = df['week_start'].dt.isocalendar().week.to_numpy(float)
    df['week_of_year'] = woy.astype(np.int16)
    df['woy_sin'] = np.sin(2 * np.pi * woy / 52.0).astype(np.float32)
    df['woy_cos'] = np.cos(2 * np.pi * woy / 52.0).astype(np.float32)

    # stock features
    if 'in_stock' in df.columns:
        out = 1.0 - df['in_stock']                                   # NaN where unknown
        for w in (4, 13):
            df[f'stockout_rate_{w}'] = _grolling(out, keys, w, 'mean').astype(np.float32)
        last_so = pd.Series(np.where(df['in_stock'] < 1, df['age_weeks'], np.nan)).groupby(keys, sort=False).ffill().to_numpy()
        df['weeks_since_stockout'] = (df['age_weeks'].to_numpy(float) - last_so).astype(np.float32)
    df['censored_rate_13'] = _grolling(df['censored'].astype(np.float32), keys, 13, 'mean').astype(np.float32)

    # MASE denominator: expanding mean |week-to-week change| (NaN until 2 weeks exist)
    absdiff = g['units'].diff().abs()
    df['mase_den'] = absdiff.groupby(keys, sort=False).expanding().mean().reset_index(level=0, drop=True).astype(np.float32)

    # scale, seasonal features, targets
    df['scale'] = np.maximum(df['roll_mean_13'].to_numpy(float), 1.0).astype(np.float32)
    for h in range(1, horizon + 1):
        df[f'seasonal_h{h}'] = (g['units'].shift(52 - h) / df['scale']).astype(np.float32)
        df[f'target_h{h}'] = (g['units'].shift(-h) / df['scale']).astype(np.float32)

    # make unit-based features scale-free
    unit_cols = [c for c in df.columns if c.startswith(('lag_', 'roll_mean_', 'roll_median_', 'roll_std_', 'roll_max_'))]
    for c in unit_cols:
        df[c] = (df[c] / df['scale']).astype(np.float32)
    df['log_scale'] = np.log1p(df['scale']).astype(np.float32)
    df['den_scaled'] = (df['mase_den'] / df['scale']).astype(np.float32)
    if 'on_hand' in df.columns:
        df['on_hand_scaled'] = (df['on_hand'] / df['scale']).astype(np.float32)

    df['is_outlier'] = False
    df['sample_weight'] = df['source'].astype(str).map(source_weights).fillna(0.35).astype(np.float32)
    print(f'Built features in {time.perf_counter() - t0:.1f}s: {len(df):,} rows x {df.shape[1]} cols')
    return df


frame = build_features(panel)
del panel
gc.collect()

display(frame.groupby('source', observed=True).agg(
    rows=('units', 'size'), series=(SID, 'nunique'), sample_weight=('sample_weight', 'first'),
    valid_h1_targets=('target_h1', lambda s: s.notna().sum()),
    median_mase_den=('mase_den', 'median')))

if SAVE_FEATURES:
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    frame.to_parquet(OUT_DIR / 'features_wide.parquet', engine='pyarrow', index=False)
    print('Saved', OUT_DIR / 'features_wide.parquet')

## 4. Targets matrix, scorer, long format

`Y` (articles × weeks) comes from the test_shop part of the panel: gaps inside an article's life are 0, weeks before its first observation are NaN, matching how `mase_den` is computed.

Score = (1 − mean per-article MASE) × 100, higher is better.

In [ ]:
Y = (frame.loc[frame['source'] == 'test_shop']
          .pivot(index='article_id', columns='week_start', values='units').sort_index(axis=1))
print(f'Test shop matrix: {Y.shape[0]:,} articles x {Y.shape[1]} weeks ({Y.columns[0].date()} -> {Y.columns[-1].date()})')

META_COLS = ['source', SID, 'week_start', 'units', 'censored', 'observed', 'store_id', 'article_id', 'scale',
             'mase_den', 'on_hand', 'is_outlier', 'sample_weight', 'supplier_id', 'product_group', 'division',
             'department', 'department_group', 'store_format', 'category', 'subcategory', 'segment']
LONG_ID_COLS = ['source', SID, 'article_id', 'week_start', 'scale', 'mase_den', 'sample_weight']
CAT_FEATURES = ['demand_class']


def rolling_cutoffs(frame, n_windows=N_WINDOWS, step=H, horizon=H):
    last = frame.loc[frame['source'] == 'test_shop', 'week_start'].max()
    return [last - pd.Timedelta(weeks=horizon + i * step) for i in range(n_windows)]


def window_weights(cutoffs):
    last = frame.loc[frame['source'] == 'test_shop', 'week_start'].max()
    same = last - pd.Timedelta(weeks=52)                     # forecasts the same calendar weeks, one year earlier
    return {c: (SAME_SEASON_WEIGHT if abs((c - same).days) <= 14 else 1.0) for c in cutoffs}


def split_frame(frame, cutoff=None, horizon=H):
    is_v = frame['source'] == 'test_shop'
    if cutoff is None:
        cutoff = frame.loc[is_v, 'week_start'].max()
    train = frame[frame['week_start'] + pd.Timedelta(weeks=horizon) <= cutoff]
    valid = frame[is_v & (frame['week_start'] == cutoff)]
    return train.reset_index(drop=True), valid.reset_index(drop=True)


def subsample_origins(frame, every=None, min_history_weeks=1):
    keep = pd.Series(True, index=frame.index)
    if every:
        k = frame['source'].astype(str).map(every).fillna(1).astype(int)
        from_end = frame.groupby(SID, sort=False, observed=True).cumcount(ascending=False)
        keep &= (from_end % k == 0)
    keep &= frame['age_weeks'] >= min_history_weeks - 1
    return frame[keep].reset_index(drop=True)


def to_long(frame, horizon=H, drop_missing_target=True):
    n = len(frame)
    feats = [c for c in frame.columns if c not in META_COLS and not c.startswith(('target_h', 'seasonal_h'))]
    base_cols = [c for c in LONG_ID_COLS if c in frame.columns] + feats
    rep = np.tile(np.arange(n, dtype=np.int64), horizon)
    long = frame[base_cols].iloc[rep].reset_index(drop=True)
    long['h'] = np.repeat(np.arange(1, horizon + 1, dtype=np.int8), n)
    long['seasonal_lag52'] = frame[[f'seasonal_h{h}' for h in range(1, horizon + 1)]].to_numpy().T.ravel()
    long['target'] = frame[[f'target_h{h}' for h in range(1, horizon + 1)]].to_numpy().T.ravel()
    long[[SID, 'article_id']] = long[[SID, 'article_id']].astype('category')
    if drop_missing_target:
        long = long[long['target'].notna()].reset_index(drop=True)
    return long


def forecast_matrix(rows, pred, index=None, horizon=H):
    index = Y.index if index is None else index
    F = pd.DataFrame({'article_id': rows['article_id'].astype(str).to_numpy(), 'h': rows['h'].to_numpy(),
                      'pred': np.asarray(pred, dtype=float)}).pivot(index='article_id', columns='h', values='pred')
    return F.reindex(index=index, columns=range(1, horizon + 1)).fillna(0).to_numpy(float)


_SCORE_PARTS = {}


def _score_parts(cutoff):
    if cutoff not in _SCORE_PARTS:
        hist = Y.loc[:, :cutoff]
        act = Y.loc[:, Y.columns > cutoff].iloc[:, :H].to_numpy(float)
        den = hist.diff(axis=1).abs().mean(axis=1).to_numpy()
        _SCORE_PARTS[cutoff] = (act, den)
    return _SCORE_PARTS[cutoff]


def per_article_mase(cutoff, F):
    act, den = _score_parts(cutoff)
    err = np.abs(act - np.clip(F, 0, None))
    with np.errstate(divide='ignore', invalid='ignore'):
        return {name: np.where(den > 0, err[:, cols].mean(axis=1) / den, np.nan)
                for name, cols in [('all', slice(0, H)), ('w1-w7', slice(0, 7)), ('w8-w13', slice(7, H))]}


def mase_score(cutoff, F):
    return {k: round((1 - np.nanmean(v)) * 100, 3) for k, v in per_article_mase(cutoff, F).items()}


def make_submission(F, index=None):
    index = Y.index if index is None else index
    sub = pd.DataFrame(np.clip(np.asarray(F, dtype=float), 0, None), index=index,
                       columns=[f'w{i}' for i in range(1, H + 1)]).reset_index()
    assert len(sub) == len(index) and sub['article_id'].is_unique and not sub.isna().any().any()
    return sub


def make_context(frame, cutoff=None, origin_stride=None, with_train=True):
    train_w, valid_w = split_frame(frame, cutoff)
    cutoff_ts = valid_w['week_start'].max()
    valid = to_long(valid_w, drop_missing_target=False)
    at_cut = valid_w.assign(article_id=valid_w['article_id'].astype(str)).set_index('article_id').reindex(Y.index)
    return {
        'cutoff': cutoff_ts,
        'train': to_long(subsample_origins(train_w, every=origin_stride)) if with_train else None,
        'valid': valid,
        'features': [c for c in valid.columns if c not in LONG_ID_COLS and c != 'target'],
        'Y_hist': Y.loc[:, :cutoff_ts],
        'weeks_since_sale': at_cut['weeks_since_sale'].to_numpy(float),
        'demand_class': at_cut['demand_class'].astype(str).to_numpy(),
    }


def train_weight(tr, den_floor=DEN_FLOOR, cap_q=WEIGHT_CAP_Q):
    # L1 on target=units/scale with weight scale/den  ==  |error in units| / den  ==  per-article MASE
    w = tr['sample_weight'].to_numpy(float) * tr['scale'].to_numpy(float) / np.clip(tr['mase_den'].to_numpy(float), den_floor, None)
    w = np.where(tr['mase_den'].to_numpy(float) > 0, w, 0.0)     # articles the scorer can't score
    return np.clip(w, 0, np.quantile(w[w > 0], cap_q) if (w > 0).any() else None)

## 5. Models (all tree / linear learners use the MASE weights)

In [ ]:
class Fitted:
    def __init__(self, predict, model=None, info=None):
        self.predict, self.model, self.info = predict, model, (info or {})


def _fitted(ctx, predict, model=None, info=None):
    return predict(ctx), Fitted(predict, model, info)


def _repeat(f):
    return np.repeat(np.asarray(f, dtype=float)[:, None], H, axis=1)


def _global_predictor(model, feats, transform=None, prep=None):
    def predict(c):
        va = c['valid']
        out = model.predict(prep(va[feats]) if prep is not None else va[feats])
        if transform is not None:
            out = transform(out)
        return forecast_matrix(va, np.clip(out, 0, None) * va['scale'].to_numpy())
    return predict


# ---------- reference models ----------
def fit_median(ctx, weeks=13):
    return _fitted(ctx, lambda c: _repeat(c['Y_hist'].iloc[:, -weeks:].median(axis=1).fillna(0)))


def fit_naive(ctx):
    return _fitted(ctx, lambda c: _repeat(c['Y_hist'].iloc[:, -1].fillna(0)))


def fit_seasonal_naive(ctx, season=52):
    def predict(c):
        Yh = c['Y_hist']
        F = Yh.iloc[:, -season:-season + H].to_numpy(float) if Yh.shape[1] >= season else np.full((len(Yh), H), np.nan)
        fallback = _repeat(Yh.iloc[:, -13:].median(axis=1).fillna(0))
        return np.where(np.isnan(F), fallback, F)
    return _fitted(ctx, predict)


# ---------- LightGBM ----------
LGB_PARAMS = dict(objective='l1', learning_rate=0.05, num_leaves=63, min_data_in_leaf=500, max_bin=63,
                  feature_fraction=0.8, bagging_fraction=0.7, bagging_freq=1, lambda_l2=2.0,
                  force_col_wise=True, n_jobs=-1, verbose=-1, seed=SEED)
LGB_TWEEDIE_PARAMS = {**LGB_PARAMS, 'objective': 'tweedie', 'tweedie_variance_power': 1.3, 'seed': SEED + 1}


def _lgb_ds(X, y, w, feats, ref=None):
    return lgb.Dataset(X, y, weight=w, categorical_feature=[c for c in CAT_FEATURES if c in feats],
                       reference=ref, free_raw_data=False)


def fit_lightgbm(ctx, params=LGB_PARAMS, rounds=LGB_ROUNDS, max_rounds=2000):
    tr, feats = ctx['train'], ctx['features']
    y = tr['target'].clip(lower=0) if params.get('objective') == 'tweedie' else tr['target']
    w = train_weight(tr)
    if rounds == 'auto':
        inner = ctx['cutoff'] - pd.Timedelta(weeks=H)
        is_tr = (tr['week_start'] + pd.Timedelta(weeks=H) <= inner).to_numpy()
        is_va = ((tr['source'] == 'test_shop') & (tr['week_start'] == inner)).to_numpy()
        if is_va.sum() == 0 or is_tr.sum() == 0:
            rounds = 300
        else:
            d_tr = _lgb_ds(tr.loc[is_tr, feats], y[is_tr], w[is_tr], feats)
            d_va = _lgb_ds(tr.loc[is_va, feats], y[is_va], w[is_va], feats, ref=d_tr)
            m = lgb.train({**params, 'metric': 'l1'}, d_tr, max_rounds, valid_sets=[d_va],
                          callbacks=[lgb.early_stopping(100, verbose=False)])
            rounds = max(50, int(m.best_iteration * 1.1))
    model = lgb.train(params, _lgb_ds(tr[feats], y, w, feats), num_boost_round=int(rounds))
    return _fitted(ctx, _global_predictor(model, feats), model, {'rounds': int(rounds)})


def fit_lightgbm_tweedie(ctx, rounds=LGB_ROUNDS):
    return fit_lightgbm(ctx, params=LGB_TWEEDIE_PARAMS, rounds=rounds)


# ---------- CatBoost (MAE) ----------
def fit_catboost(ctx, iterations=550, depth=6, learning_rate=0.07, l2_leaf_reg=5.0):
    tr, feats = ctx['train'], ctx['features']
    cat_cols = [c for c in CAT_FEATURES if c in feats]

    def prep(df):
        out = df.copy()
        for c in cat_cols:
            out[c] = out[c].astype(str)
        return out

    model = CatBoostRegressor(loss_function='MAE', eval_metric='MAE', iterations=iterations, depth=depth,
                              learning_rate=learning_rate, l2_leaf_reg=l2_leaf_reg, random_seed=SEED,
                              thread_count=-1, verbose=0)
    model.fit(prep(tr[feats]), tr['target'], cat_features=cat_cols, sample_weight=train_weight(tr))
    return _fitted(ctx, _global_predictor(model, feats, prep=prep), model)


# ---------- XGBoost (L1, GPU if available) ----------
def fit_xgb(ctx, n_estimators=700, learning_rate=0.05, max_depth=7):
    tr, feats = ctx['train'], ctx['features']
    model = xgb.XGBRegressor(device=XGB_DEVICE, tree_method='hist', objective='reg:absoluteerror',
                             n_estimators=n_estimators, learning_rate=learning_rate, max_depth=max_depth,
                             min_child_weight=50, subsample=0.8, colsample_bytree=0.8, reg_lambda=2.0,
                             enable_categorical=True, max_cat_to_onehot=8, random_state=SEED)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        model.fit(tr[feats], tr['target'], sample_weight=train_weight(tr))

    def predict(c):
        va = c['valid']
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            out = model.predict(va[feats])
        return forecast_matrix(va, np.clip(out, 0, None) * va['scale'].to_numpy())
    return _fitted(ctx, predict, model)


# ---------- HistGradientBoosting (L1) ----------
def fit_hist_gbdt(ctx, max_iter=300, learning_rate=0.06, max_leaf_nodes=63, min_samples_leaf=500):
    tr, feats = ctx['train'], ctx['features']
    cat_cols = [c for c in CAT_FEATURES if c in feats]
    num_cols = [c for c in feats if c not in cat_cols]
    pre = ColumnTransformer([('num', 'passthrough', num_cols),
                             ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), cat_cols)])
    cat_idx = list(range(len(num_cols), len(num_cols) + len(cat_cols)))
    model = make_pipeline(pre, HistGradientBoostingRegressor(
        loss='absolute_error', learning_rate=learning_rate, max_iter=max_iter, max_leaf_nodes=max_leaf_nodes,
        min_samples_leaf=min_samples_leaf, l2_regularization=2.0,
        categorical_features=cat_idx or None, random_state=SEED))
    model.fit(tr[feats], tr['target'], histgradientboostingregressor__sample_weight=train_weight(tr))
    return _fitted(ctx, _global_predictor(model, feats), model)


# ---------- linear L1 ----------
def signed_log(x):
    return np.sign(x) * np.log1p(np.abs(x))


def make_tabular_preprocessor(features, categorical=(*CAT_FEATURES, 'h')):
    cat = [c for c in categorical if c in features]
    num = [c for c in features if c not in cat]
    return ColumnTransformer([
        ('num', make_pipeline(FunctionTransformer(signed_log), SimpleImputer(strategy='median', add_indicator=True),
                              StandardScaler()), num),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat)])


def fit_linear_l1(ctx, sample=600_000, alpha=1e-5, epochs=12, target_clip_q=0.999):
    tr = ctx['train']
    tr = tr.sample(n=sample, random_state=SEED) if len(tr) > sample else tr
    feats = ctx['features']
    model = make_pipeline(make_tabular_preprocessor(feats),
                          SGDRegressor(loss='epsilon_insensitive', epsilon=0.0, alpha=alpha, average=True,
                                       max_iter=epochs, tol=None, random_state=SEED))
    y = tr['target'].clip(upper=tr['target'].quantile(target_clip_q))
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        model.fit(tr[feats], y, sgdregressor__sample_weight=train_weight(tr))
    return _fitted(ctx, _global_predictor(model, feats), model)


# ---------- TSB & Croston SBA ----------
def tsb_state(Y_hist, alpha=0.1, beta=0.05):
    X = Y_hist.to_numpy(float)
    p, z = np.zeros(len(X)), np.full(len(X), np.nan)
    for t in range(X.shape[1]):
        d = X[:, t]
        known = ~np.isnan(d)
        sale = known & (d > 0)
        z = np.where(sale, np.where(np.isnan(z), d, z + alpha * (d - z)), z)
        p = np.where(known, p + beta * (sale - p), p)
    return p, np.nan_to_num(z)


def fit_tsb(ctx, alpha=0.1, beta=0.05, mode='median', threshold=0.5):
    def predict(c):
        p, z = tsb_state(c['Y_hist'], alpha, beta)
        return _repeat(p * z if mode == 'mean' else np.where(p > threshold, z, 0.0))
    return _fitted(ctx, predict)


def croston_sba_state(Y_hist, alpha=0.1):
    X = Y_hist.to_numpy(float)
    n = len(X)
    z, p, q = np.full(n, np.nan), np.full(n, np.nan), np.ones(n)
    for t in range(X.shape[1]):
        d = X[:, t]
        known = ~np.isnan(d)
        sale = known & (d > 0)
        z = np.where(sale, np.where(np.isnan(z), d, z + alpha * (d - z)), z)
        p = np.where(sale, np.where(np.isnan(p), q, p + alpha * (q - p)), p)
        q = np.where(sale, 1.0, np.where(known, q + 1.0, q))
    return np.where((p > 0) & np.isfinite(z), (1.0 - alpha / 2.0) * z / p, 0.0)


def fit_croston_sba(ctx, alpha=0.1):
    return _fitted(ctx, lambda c: _repeat(croston_sba_state(c['Y_hist'], alpha)))

## 6. Post-processing

In [ ]:
def zero_no_sales_class(F, ctx, classes=('no_sales',)):
    return np.where(np.isin(ctx['demand_class'], classes)[:, None], 0.0, F)


def dead_to_zero(F, ctx, weeks=8):
    dead = ~(ctx['weeks_since_sale'] < weeks)          # NaN (never sold / no row) counts as dead
    return np.where(dead[:, None], 0.0, F)


def snap_small(F, ctx, below=0.5):
    return np.where(F < below, 0.0, F)


def postprocess(F, ctx, steps):
    F = np.clip(F, 0, None)
    for step in (steps or ()):
        func, kw = (step, {}) if callable(step) else step
        F = func(F, ctx, **kw)
    return F

## 7. Model lineup, ensembles, router, blend ← edit here

In [ ]:
MODELS = {
    'median_13w':        (fit_median, {'weeks': 13}),
    'naive_last':        (fit_naive, {}),
    'seasonal_naive_52': (fit_seasonal_naive, {}),
    'lightgbm':          (fit_lightgbm, {}),
    'lightgbm_tweedie':  (fit_lightgbm_tweedie, {'rounds': 300}),
    'tsb':               (fit_tsb, {'alpha': 0.1, 'beta': 0.05, 'mode': 'median'}),
    'croston_sba':       (fit_croston_sba, {'alpha': 0.1}),
}
if HAS_CATBOOST:
    MODELS['catboost'] = (fit_catboost, {'iterations': 400 if FAST_MODE else 550})
if USE_XGB:
    MODELS['xgb_l1'] = (fit_xgb, {'n_estimators': 500 if FAST_MODE else 800})
if not FAST_MODE:
    MODELS['hist_gbdt'] = (fit_hist_gbdt, {'max_iter': 300})
    MODELS['linear_l1'] = (fit_linear_l1, {'sample': 1_200_000})

REFERENCES = ['median_13w', 'naive_last', 'seasonal_naive_52']
BASELINE = 'median_13w'

tree_members = [m for m in ['lightgbm', 'catboost', 'xgb_l1'] if m in MODELS]
diverse_members = [m for m in ['lightgbm', 'lightgbm_tweedie', 'catboost', 'xgb_l1', 'hist_gbdt', 'linear_l1', 'tsb'] if m in MODELS]
ENSEMBLES = {
    'ens_mean(trees)':         (np.mean, tree_members),
    'ens_median(top_diverse)': (np.median, diverse_members),
}

CLASS_ROUTED_NAME = 'class_routed'
CLASS_ROUTE_MAP = {'smooth': 'ens_mean(trees)', 'erratic': 'ens_mean(trees)',
                   'intermittent': 'tsb', 'lumpy': 'tsb', 'no_sales': 'tsb'}

# Weighted blend: weights + one multiplier tuned on the backtest windows
BLEND_NAME = 'blend'
BLEND_MEMBERS = [m for m in ['lightgbm', 'catboost', 'xgb_l1', 'tsb', 'median_13w'] if m in MODELS]
BLEND_STEP = 0.1 if len(BLEND_MEMBERS) <= 4 else 0.2
CALIBRATION_GRID = np.round(np.arange(0.70, 1.31, 0.05), 2)

DEFAULT_POSTPROCESS = [(zero_no_sales_class, {'classes': ('no_sales',)}), (dead_to_zero, {'weeks': 8}), (snap_small, {'below': 0.5})]
MODEL_POSTPROCESS = {
    'tsb': [(zero_no_sales_class, {'classes': ('no_sales',)}), (dead_to_zero, {'weeks': 10})],
    'croston_sba': [(zero_no_sales_class, {'classes': ('no_sales',)}), (dead_to_zero, {'weeks': 10}), (snap_small, {'below': 0.4})],
}
ENSEMBLE_POSTPROCESS = {
    CLASS_ROUTED_NAME: [(zero_no_sales_class, {'classes': ('no_sales',)}), (dead_to_zero, {'weeks': 9}), (snap_small, {'below': 0.35})],
    BLEND_NAME: DEFAULT_POSTPROCESS,
}


def get_post_steps(name):
    if name in REFERENCES:
        return []
    return ENSEMBLE_POSTPROCESS.get(name, MODEL_POSTPROCESS.get(name, DEFAULT_POSTPROCESS))


print('Models:', list(MODELS))
print('Blend members:', BLEND_MEMBERS, '| step', BLEND_STEP)

## 8. Rolling backtest

Four windows ending 13, 26, 39 and 52 weeks before the shop's last week. The last one forecasts the same calendar weeks as the real submission and counts `SAME_SEASON_WEIGHT` times.

In [ ]:
def route_by_demand_class(fc, ctx, route_map=CLASS_ROUTE_MAP):
    dc = ctx['demand_class']
    out = fc[route_map.get('smooth', 'lightgbm')].copy()
    for cls_name, model_name in route_map.items():
        if model_name in fc:
            mask = dc == cls_name
            out[mask] = fc[model_name][mask]
    return out


def add_combinations(fc, ctx):
    for name, (how, members) in ENSEMBLES.items():
        if members and all(m in fc for m in members):
            fc[name] = how(np.stack([fc[m] for m in members]), axis=0)
    if all(m in fc for m in set(CLASS_ROUTE_MAP.values())):
        fc[CLASS_ROUTED_NAME] = route_by_demand_class(fc, ctx)
    return fc


cutoffs = rolling_cutoffs(frame)
WINDOW_W = window_weights(cutoffs)
print('Windows (cutoff = last known week):', {c.date(): w for c, w in WINDOW_W.items()})

forecasts, contexts, fitted, fit_info = {}, {}, {}, []
for i, cutoff in enumerate(cutoffs):
    t0 = time.perf_counter()
    ctx = make_context(frame, cutoff, ORIGIN_STRIDE)
    contexts[cutoff] = {k: v for k, v in ctx.items() if k not in ('train', 'valid')}
    print(f'\n=== cutoff {cutoff.date()} | train {len(ctx["train"]):,} rows '
          f'{ctx["train"]["source"].value_counts().to_dict()} | context {time.perf_counter() - t0:.1f}s')
    fc = {}
    for name, (fit, kw) in MODELS.items():
        t0 = time.perf_counter()
        fc[name], model = fit(ctx, **kw)
        if i == 0:
            fitted[name] = model
        sc = mase_score(cutoff, postprocess(fc[name], ctx, get_post_steps(name)))
        extra = f" rounds={model.info['rounds']}" if 'rounds' in model.info else ''
        fit_info.append({'model': name, 'cutoff': cutoff.date(), **model.info})
        print(f'  {name:<24s} all {sc["all"]:8.2f} | w8-w13 {sc["w8-w13"]:8.2f}   ({time.perf_counter() - t0:.1f}s){extra}')
    fc = add_combinations(fc, ctx)
    for name, F in fc.items():
        forecasts[(name, cutoff)] = F
    del ctx
    gc.collect()

### Blend: tune weights + multiplier

Final weights are tuned on all windows. The blend's **backtest score** uses leave-one-window-out tuning (each window scored with weights picked on the other windows), so it isn't flattered by being tuned on the weeks it's scored on.

In [ ]:
def simplex_grid(k, step):
    n = int(round(1 / step))
    for combo in itertools.product(range(n + 1), repeat=k - 1):
        if sum(combo) <= n:
            yield np.array(list(combo) + [n - sum(combo)], dtype=float) / n


def blend_forecast(cutoff, w, k=1.0, members=BLEND_MEMBERS):
    return k * sum(wi * forecasts[(m, cutoff)] for wi, m in zip(w, members))


def tune_blend(cuts):
    wsum = sum(WINDOW_W[c] for c in cuts)
    score = lambda F_by_c: sum(WINDOW_W[c] * mase_score(c, postprocess(F_by_c[c], contexts[c], get_post_steps(BLEND_NAME)))[SELECT_METRIC]
                               for c in cuts) / wsum
    best_w, best_s = None, -np.inf
    for w in simplex_grid(len(BLEND_MEMBERS), BLEND_STEP):
        s = score({c: blend_forecast(c, w) for c in cuts})
        if s > best_s:
            best_w, best_s = w, s
    best_k, best_s = 1.0, -np.inf
    for k in CALIBRATION_GRID:
        s = score({c: blend_forecast(c, best_w, k) for c in cuts})
        if s > best_s:
            best_k, best_s = k, s
    return best_w, best_k, best_s


t0 = time.perf_counter()
BLEND_W, BLEND_K, s_all = tune_blend(cutoffs)
print(f'Final blend (all windows): {dict(zip(BLEND_MEMBERS, BLEND_W.round(2)))} | multiplier {BLEND_K} | in-sample {SELECT_METRIC} {s_all:.2f}')

for c in cutoffs:                                   # leave-one-window-out forecasts for honest scoring
    others = [o for o in cutoffs if o != c]
    w, k, _ = tune_blend(others) if others else (BLEND_W, BLEND_K, None)
    forecasts[(BLEND_NAME, c)] = blend_forecast(c, w, k)
    print(f'  held-out {c.date()}: weights {dict(zip(BLEND_MEMBERS, w.round(2)))} k={k}')
print(f'({time.perf_counter() - t0:.1f}s)')

## 9. Results (window-weighted)

In [ ]:
rows = []
for (name, c), F in forecasts.items():
    rows.append({'model': name, 'cutoff': c, **mase_score(c, postprocess(F, contexts[c], get_post_steps(name)))})
results = pd.DataFrame(rows)


def wmean(df, col):
    return df.groupby('model').apply(lambda d: np.average(d[col], weights=[WINDOW_W[c] for c in d['cutoff']]),
                                     include_groups=False)


summary = pd.DataFrame({col: wmean(results, col) for col in ['all', 'w1-w7', 'w8-w13']})
per_window = results.pivot(index='model', columns='cutoff', values='all')
per_window.columns = [f'{c.date()}{" (same season)" if WINDOW_W[c] > 1 else ""}' for c in per_window.columns]
base = results[results.model == BASELINE].set_index('cutoff')['all']
summary['beats baseline (# windows)'] = results.groupby('model').apply(
    lambda d: int((d.set_index('cutoff')['all'] > base.reindex(d['cutoff'])).sum()), include_groups=False)
summary = summary.sort_values(SELECT_METRIC, ascending=False)
print(f'Window-weighted scores (same-season window x{SAME_SEASON_WEIGHT}), sorted by {SELECT_METRIC}:')
display(summary.round(2))
print('Per window (all w1-w13):')
display(per_window.loc[summary.index].round(2))
if fit_info and any('rounds' in r for r in fit_info):
    display(pd.DataFrame(fit_info).dropna(axis=1, how='all').pivot(index='model', columns='cutoff', values='rounds').dropna(how='all'))

## 10. Diagnostics

In [ ]:
c0 = cutoffs[0]                       # most recent window
best_name = summary.index[~summary.index.isin(REFERENCES)][0]
G = postprocess(forecasts[(best_name, c0)], contexts[c0], get_post_steps(best_name))
m = pd.Series(per_article_mase(c0, G)['all'], index=Y.index).dropna().sort_values(ascending=False)
share = m.cumsum() / m.sum()
print(f'{best_name} on {c0.date()}: {len(m):,} scored articles')
for q in (0.01, 0.05, 0.10, 0.25):
    print(f'  top {q:>4.0%} of articles -> {share.iloc[max(int(len(m) * q) - 1, 0)]:.0%} of total MASE')
worst = m.head(15).to_frame('mase')
worst['demand_class'] = pd.Series(contexts[c0]['demand_class'], index=Y.index).reindex(worst.index)
worst['weeks_since_sale'] = pd.Series(contexts[c0]['weeks_since_sale'], index=Y.index).reindex(worst.index)
worst['hist_mean'] = Y.loc[worst.index, :c0].mean(axis=1)
worst['future_mean'] = Y.loc[worst.index, Y.columns > c0].iloc[:, :H].mean(axis=1)
worst['forecast_mean'] = pd.DataFrame(G, index=Y.index).loc[worst.index].mean(axis=1)
display(worst.round(2))

by_class = {}
dc = contexts[c0]['demand_class']
for name in summary.index:
    G_ = postprocess(forecasts[(name, c0)], contexts[c0], get_post_steps(name))
    by_class[name] = pd.Series(per_article_mase(c0, G_)['all']).groupby(dc).mean()
cls = pd.DataFrame(by_class).T
cls.loc['articles'] = pd.Series(dc).value_counts()
print('Mean MASE by demand class (lower is better):')
display(cls[[c for c in ['smooth', 'erratic', 'intermittent', 'lumpy', 'no_sales'] if c in cls]].round(3))

if 'lightgbm' in fitted:
    mdl = fitted['lightgbm'].model
    imp = pd.Series(mdl.feature_importance('gain'), index=mdl.feature_name()).sort_values()
    imp = (imp / imp.sum() * 100).tail(25)
    fig, ax = plt.subplots(figsize=(9, 7))
    ax.barh(imp.index, imp.values, color=BLUE, height=0.6)
    ax.grid(axis='y', visible=False)
    ax.set_xlabel('% of total gain')
    ax.set_title(f'LightGBM: top 25 features by gain (window {c0.date()})')
    plt.tight_layout(); plt.show()

## 11. Final fit & submission

Picks the best non-reference candidate by the window-weighted `SELECT_METRIC`, refits the base models it needs on all history, and writes `submissions/submission_<model>.csv`.

In [ ]:
candidates = [m for m in summary.index if m not in REFERENCES]
FINAL_MODEL = candidates[0]
SAVE = True


def required_base_models(name):
    if name == BLEND_NAME:
        return [m for m, w in zip(BLEND_MEMBERS, BLEND_W) if w > 0]
    if name == CLASS_ROUTED_NAME:
        return sorted({b for t in set(CLASS_ROUTE_MAP.values()) for b in required_base_models(t)})
    if name in ENSEMBLES:
        return list(ENSEMBLES[name][1])
    return [name]


members = required_base_models(FINAL_MODEL)
print(f'Final model: {FINAL_MODEL} | base models: {members}')
print(f'Backtest ({SELECT_METRIC}, window-weighted): {summary.loc[FINAL_MODEL, SELECT_METRIC]:.2f} '
      f'vs baseline {summary.loc[BASELINE, SELECT_METRIC]:.2f}')

ctx_final = make_context(frame, cutoff=None, origin_stride=ORIGIN_STRIDE)
print(f'Training rows: {len(ctx_final["train"]):,} | forecast origin: {ctx_final["cutoff"].date()}')
final_fc = {}
for name in members:
    t0 = time.perf_counter()
    fit, kw = MODELS[name]
    final_fc[name], mdl = fit(ctx_final, **kw)
    print(f'  fitted {name} ({time.perf_counter() - t0:.1f}s){" rounds=" + str(mdl.info["rounds"]) if "rounds" in mdl.info else ""}')

if FINAL_MODEL == BLEND_NAME:
    F_raw = BLEND_K * sum(w * final_fc[m] for m, w in zip(BLEND_MEMBERS, BLEND_W) if w > 0)
else:
    final_fc = add_combinations(final_fc, ctx_final)
    F_raw = final_fc[FINAL_MODEL]

F_final = postprocess(F_raw, ctx_final, get_post_steps(FINAL_MODEL))
sub = make_submission(F_final)
print(f'Articles forecast 0 for all 13 weeks: {(F_final.sum(axis=1) == 0).sum():,} of {len(sub):,}')
print(f'Mean forecast per article-week: {F_final.mean():.2f} (last 13 weeks of history: {np.nanmean(Y.iloc[:, -13:].to_numpy()):.2f})')
display(sub.head())

if SAVE:
    SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)
    path = SUBMISSION_DIR / f'submission_{re.sub(r"[^A-Za-z0-9]+", "_", FINAL_MODEL).strip("_")}.csv'
    sub.to_csv(path, index=False)
    print('Saved', path)
    if IN_COLAB:
        try:
            from google.colab import files
            files.download(str(path))
        except Exception as e:
            print('Auto-download skipped:', e)

## 12. All model scores

In [ ]:
bars = summary[SELECT_METRIC].sort_values()
fig, ax = plt.subplots(figsize=(10, 0.42 * len(bars) + 1.4))
for y, (name, v) in enumerate(bars.items()):
    ax.barh(y, v, height=0.62, color=GREY if name in REFERENCES else BLUE)
    ax.text(v, y, f' {v:.1f}', va='center', ha='left' if v >= 0 else 'right', color=INK, fontsize=9)
ax.set_yticks(range(len(bars)), [f'{n} (final)' if n == FINAL_MODEL else n for n in bars.index])
ax.axvline(0, color=INK, linewidth=0.8)
ax.grid(axis='y', visible=False)
ax.set_xlabel(f'score = (1 - MASE) x 100 on {SELECT_METRIC}, window-weighted (higher is better)')
ax.set_title('Model scores across backtest windows (grey = reference forecasts)')
plt.tight_layout(); plt.show()